## Import Files

In [ ]:
import pandas as pd
import numpy as np
import json
from mstr_robotics._connectors import mstr_api
import yaml
from mstrio.connection import Connection
from mstrio.api import browsing
from mstr_robotics.dossier import doss_read_out,doss_read_out_det
from mstr_robotics.read_out_prj_obj import read_gen
from mstr_robotics.user_RAG import  perplexity
from dotenv import load_dotenv
i_mstr_api=mstr_api()
u_perplexity=perplexity()
env_file="..\\config\\streamlit.env"
osi_dashboard="C:/coding/Python_environments/mstr_robotics/import_files/OSI_daashboard"

load_dotenv(env_file)
i_read_gen=read_gen()
with open('..\\config\\user_d.json', 'r', encoding='utf-8') as openfile:
    user_d = json.load(openfile)

with open('C:\\coding\\Python_environments\\OSI_Files\\osi-schema-with-dashboards.json', 'r', encoding='utf-8') as openfile:
    osi_dashboard_schema_d = json.load(openfile)

def parse_ds_obj(obj_def_d):
    rows_l = []
    for ds in obj_def_d["datasets"]:
        ds_id   = ds["id"]
        ds_name = ds["name"]

        row_ids    = {o["id"] for o in ds.get("rows", [])}
        pageby_ids = {o["id"] for o in ds.get("pageBy", [])}
        col_ids    = set()
        for c in ds.get("columns", []):
            if c["type"] == "templateMetrics":
                for e in c.get("elements", []):
                    col_ids.add(e["id"])
            else:
                col_ids.add(c["id"])

        for obj in ds.get("availableObjects", []):
            placement = []
            if obj["id"] in row_ids:    placement.append("rows")
            if obj["id"] in col_ids:    placement.append("columns")
            if obj["id"] in pageby_ids: placement.append("pageBy")
            placement_str = ",".join(placement) if placement else "available"

            base = {
                "dataset_id":   ds_id,
                "dataset_name": ds_name,
                "obj_id":       obj["id"],
                "obj_name":     obj["name"],
                "obj_type":     obj["type"],
                "placement":    placement_str,
            }
            forms = obj.get("forms", [])
            if forms:
                for f in forms:
                    rows_l.append({**base,
                        "form_id":          f["id"],
                        "form_name":        f["name"],
                        "dataType":         f["dataType"],
                        "baseFormCategory": f["baseFormCategory"],
                        "baseFormType":     f["baseFormType"],
                    })
            else:
                rows_l.append({**base,
                    "form_id": None, "form_name": None,
                    "dataType": None, "baseFormCategory": None, "baseFormType": None,
                })
    return pd.DataFrame(rows_l)

def fetch_json_search(conn,search_instance_resp,limit=100):
    dpn_count=search_instance_resp.json()["totalItems"]
    try:
        if dpn_count>0:
            full_result_d_l=[]
            fetched_items_count=0
            while fetched_items_count <dpn_count:
                search_d_l=browsing.get_search_results(
                connection= conn,
                search_id=search_instance_resp.json()["id"],
                project_id=conn.project_id,
                offset=fetched_items_count,
                limit=limit).json()
                full_result_d_l.extend(search_d_l)
                fetched_items_count+=limit
    except Exception as e:
            print(e)
    return full_result_d_l

def read_out_obj(conn,search_instance_resp,serch_obj): 
    osi_list=fetch_json_search(conn,search_instance_resp,limit=100)
    ol=[]
    for o in osi_list:
        if o["subtype"] in[3072,768,1024,3840,3328]:
            oo={}
            oo["id"]=o["id"]
            oo["type"]=o["type"]
            oo["subtype"]=o["subtype"]
            ol.append(oo)

    ol.append(serch_obj)

    full_obj_def_l=[]
    for o in ol:
        obj_def_d=i_read_gen.get_obj_def(conn=conn,
                                        object_id=o["id"],
                                        obj_type=o["type"],
                                        obj_sub_type=o["subtype"])
        full_obj_def_l.append(obj_def_d)
    return full_obj_def_l

def _mstr_ext(data_d):
    def _np_default(o):
        if isinstance(o, np.integer):  return int(o)
        if isinstance(o, np.floating): return float(o)
        if isinstance(o, np.bool_):    return bool(o)
        if isinstance(o, np.ndarray):  return o.tolist()
        raise TypeError(f"Object of type {type(o).__name__} is not JSON serializable")
    return {"vendor_name": "MSTR_ROBOTICS", "data": json.dumps(data_d, default=_np_default)}


def _build_osi_field(attr_rows):
    first = attr_rows.iloc[0]
    forms = [
        {k: r[k] for k in ("form_id", "form_name", "dataType", "baseFormCategory", "baseFormType")}
        for _, r in attr_rows.iterrows()
        if r["form_id"] is not None
    ]
    return {
        "name": first["obj_name"],
        "expression": {"dialects": [{"dialect": "MSTR_ROBOTICS", "expression": first["obj_id"]}]},
        "custom_extensions": [_mstr_ext({
            "obj_id": first["obj_id"], "obj_type": first["obj_type"], "forms": forms,
        })],
    }


def _build_osi_metric_field(row):
    return {
        "name": row["obj_name"],
        "expression": {"dialects": [{"dialect": "MSTR_ROBOTICS", "expression": row["obj_id"]}]},
        "custom_extensions": [_mstr_ext({"obj_id": row["obj_id"], "obj_type": row["obj_type"]})],
    }


def _build_osi_dataset(ds_rows):
    first = ds_rows.iloc[0]
    attr_fields = [
        _build_osi_field(grp.reset_index(drop=True))
        for _, grp in ds_rows[ds_rows["obj_type"] == "attribute"].groupby("obj_id", sort=False)
    ]
    metric_fields = [
        _build_osi_metric_field(r)
        for _, r in ds_rows[ds_rows["obj_type"] == "metric"].drop_duplicates(subset="obj_id").iterrows()
    ]
    fields = attr_fields + metric_fields
    dataset = {
        "name":   first["dataset_name"],
        "source": first["dataset_id"],
        "custom_extensions": [_mstr_ext({
            "dataset_id": first["dataset_id"], "dataset_name": first["dataset_name"],
        })],
    }
    if fields:
        dataset["fields"] = fields
    return dataset


def _build_osi_metrics(df):
    return [
        {
            "name": r["obj_name"],
            "expression": {"dialects": [{"dialect": "MSTR_ROBOTICS", "expression": r["obj_id"]}]},
            "custom_extensions": [_mstr_ext({"obj_id": r["obj_id"], "obj_type": r["obj_type"]})],
        }
        for _, r in df[df["obj_type"] == "metric"].drop_duplicates(subset="obj_id").iterrows()
    ]


def _build_osi_relationships(obj_def_d):
    rel_df = find_dataset_relationships(obj_def_d)
    if rel_df.empty:
        return []
    rels = []
    for (from_id, to_id), grp in rel_df.groupby(["from_dataset_id", "to_dataset_id"], sort=False):
        first      = grp.iloc[0]
        from_name  = first["from_dataset_name"]
        to_name    = first["to_dataset_name"]
        attr_names = grp["attr_name"].tolist()
        join_keys  = [
            {"attr_name": r["attr_name"], "attr_id": r["attr_id"]}
            for _, r in grp.iterrows()
        ]
        rels.append({
            "name":         f"{from_name}__{to_name}",
            "from":         from_name,
            "to":           to_name,
            "from_columns": attr_names,
            "to_columns":   list(attr_names),
            "custom_extensions": [_mstr_ext({
                "from_dataset_id":   from_id,
                "from_dataset_name": from_name,
                "to_dataset_id":     to_id,
                "to_dataset_name":   to_name,
                "join_keys":         join_keys,
            })],
        })
    return rels

def find_dataset_relationships(obj_def_d):
    """
    Return one row per (dataset-pair, shared-attribute).
    Direction: dataset appearing earlier in obj_def_d["datasets"] is 'from'.
    """
    df = parse_ds_obj(obj_def_d)
    attr_df = (
        df[df["obj_type"] == "attribute"][["dataset_id", "dataset_name", "obj_id", "obj_name"]]
        .drop_duplicates()
    )
    ds_order = {ds["id"]: i for i, ds in enumerate(obj_def_d["datasets"])}
    shared_attr = (
        attr_df.groupby("obj_id")
        .filter(lambda g: g["dataset_id"].nunique() > 1)
    )
    rels = []
    for obj_id, grp in shared_attr.groupby("obj_id", sort=False):
        ds_list = (
            grp.drop_duplicates("dataset_id")
            .assign(order=lambda x: x["dataset_id"].map(ds_order))
            .sort_values("order")
            .to_dict("records")
        )
        attr_name = ds_list[0]["obj_name"]
        for i in range(len(ds_list)):
            for j in range(i + 1, len(ds_list)):
                rels.append({
                    "from_dataset_id":   ds_list[i]["dataset_id"],
                    "from_dataset_name": ds_list[i]["dataset_name"],
                    "to_dataset_id":     ds_list[j]["dataset_id"],
                    "to_dataset_name":   ds_list[j]["dataset_name"],
                    "attr_id":           obj_id,
                    "attr_name":         attr_name,
                })
    return pd.DataFrame(rels) if rels else pd.DataFrame(
        columns=["from_dataset_id","from_dataset_name","to_dataset_id","to_dataset_name","attr_id","attr_name"]
    )

def build_osi_semantic_model(obj_def_d):
    df            = parse_ds_obj(obj_def_d)
    datasets      = [_build_osi_dataset(grp.reset_index(drop=True))
                     for _, grp in df.groupby("dataset_id", sort=False)]
    relationships = _build_osi_relationships(obj_def_d)
    metrics       = _build_osi_metrics(df)

    semantic_model = {
        "name":     obj_def_d.get("name", "unknown"),
        "datasets": datasets,
    }
    if relationships:
        semantic_model["relationships"] = relationships
    if metrics:
        semantic_model["metrics"] = metrics

    return {"version": "0.2.0", "semantic_model": [semantic_model]}

###############

def _map_viz_type(mstr_type):
    return MSTR_VIZ_TYPE_MAP.get(mstr_type, "custom")


def _build_selector_filter(grp, key_to_name):
    """One OSI Filter from all rows of a single selector (grouped by sel_filt_key)."""
    first    = grp.iloc[0]
    sel_type = first["selector_type"]
    osi_type = MSTR_SEL_TYPE_MAP.get(sel_type, "element_selection")

    if "target_key" in grp.columns:
        target_keys  = grp["target_key"].dropna().unique().tolist()
        target_names = [key_to_name.get(k, k) for k in target_keys]
    else:
        target_names = []

    filt = {"name": first["sel_filt_name"], "type": osi_type}
    if target_names:
        filt["target"] = {"scope": "visualization", "specific_targets": target_names}

    if sel_type == "attribute_element_list":
        filt["definition"] = {"attribute": first.get("target_object_name", ""), "selection_type": "include"}
    elif sel_type == "metric_qualification":
        filt["definition"] = {"expression": str(first.get("summary", "")),
                               "applies_to": {"metric": first.get("target_object_name", "")}}
    elif sel_type == "object_replacement":
        filt["definition"] = {"object_type": first.get("target_object_type", ""),
                               "available_objects": grp["target_object_name"].dropna().unique().tolist()}
    elif sel_type == "visualization_as_filter":
        filt["definition"] = {"attribute": first.get("target_object_name", ""), "selection_type": "include"}

    filt["custom_extensions"] = [_mstr_ext({
        "sel_filt_key":   first["sel_filt_key"],
        "selector_type":  sel_type,
        "display_style":  first.get("display_style", ""),
        "has_all_option": bool(first.get("has_all_option", False)),
    })]
    return filt


def _semantic_ref(obj_rows, ref_type):
    """One OSI SemanticRef with MSTR detail in custom_extensions."""
    first = obj_rows.iloc[0]
    forms = [
        {"form_id": r["form_id"], "form_name": r["form_name"]}
        for _, r in obj_rows.iterrows()
        if r.get("form_id")
    ]
    ext_data = {
        "object_id":   first["object_id"],
        "object_name": first["object_name"],
        "type":        first["type"],
        "row_col_fg":  first.get("row_col_fg"),
        "row_col_nr":  first.get("row_col_nr"),
    }
    if forms:
        ext_data["forms"] = forms
    return {
        "name":              first["object_name"],
        "type":              ref_type,
        "custom_extensions": [_mstr_ext(ext_data)],
    }


def _build_visualization(viz_rows):
    """One OSI Visualization — each SemanticRef carries its MSTR detail in custom_extensions."""
    first     = viz_rows.iloc[0]
    attr_rows = viz_rows[viz_rows["type"] == "attribute"]
    met_rows  = viz_rows[viz_rows["type"] == "metric"]

    dims = [_semantic_ref(grp.reset_index(drop=True), "field")
            for _, grp in attr_rows.groupby("object_id", sort=False)]
    mets = [_semantic_ref(grp.reset_index(drop=True), "metric")
            for _, grp in met_rows.groupby("object_id", sort=False)]

    viz = {
        "name": first["visual_name"],
        "type": _map_viz_type(first["visualizationType"]),
        "custom_extensions": [_mstr_ext({
            "visual_key":        first["visual_key"],
            "visualizationType": first["visualizationType"],
        })],
    }
    if dims: viz["dimensions"] = dims
    if mets: viz["metrics"]    = mets
    return viz


def _build_page(page_rows, page_sel_df, key_to_name):
    first        = page_rows.iloc[0]
    page_filters = []
    if not page_sel_df.empty:
        for _, grp in page_sel_df.groupby("sel_filt_key", sort=False):
            page_filters.append(_build_selector_filter(grp.reset_index(drop=True), key_to_name))
    vizs = [_build_visualization(grp.reset_index(drop=True))
            for _, grp in page_rows.groupby("visual_key", sort=False)]
    page = {"name": first["page_name"],
            "custom_extensions": [_mstr_ext({"page_key": first["page_key"]})]}
    if page_filters: page["filters"]        = page_filters
    if vizs:         page["visualizations"] = vizs
    return page


def _build_chapter(chap_rows, chap_filt_df, chap_sel_df, key_to_name):
    first        = chap_rows.iloc[0]
    chap_filters = []
    if not chap_filt_df.empty:
        for _, grp in chap_filt_df.groupby("sel_filt_key", sort=False):
            chap_filters.append(_build_selector_filter(grp.reset_index(drop=True), key_to_name))
    pages = []
    for page_key, page_rows in chap_rows.groupby("page_key", sort=False):
        page_sel = (chap_sel_df[chap_sel_df["page_key"] == page_key]
                    if not chap_sel_df.empty else pd.DataFrame())
        pages.append(_build_page(page_rows.reset_index(drop=True), page_sel, key_to_name))
    chapter = {"name": first["chapter_name"],
               "custom_extensions": [_mstr_ext({"chapter_key": first["chapter_key"]})]}
    if chap_filters: chapter["filters"] = chap_filters
    if pages:        chapter["pages"]   = pages
    return chapter


def build_osi_dashboard(doss_hier_l, filt_sel_d, semantic_model_name):
    hier_df = pd.DataFrame(doss_hier_l)
    filt_df = pd.DataFrame(filt_sel_d.get("dos_filt_d_l", []))
    sel_df  = pd.DataFrame(filt_sel_d.get("page_selector_d_l", []))
    key_to_name = (hier_df.drop_duplicates("visual_key")
                   .set_index("visual_key")["visual_name"].to_dict()
                   if not hier_df.empty else {})
    dashboards = []
    for dossier_id, doss_rows in hier_df.groupby("dossier_id", sort=False):
        first     = doss_rows.iloc[0]
        doss_filt = filt_df[filt_df["dossier_id"] == dossier_id] if not filt_df.empty else pd.DataFrame()
        doss_sel  = sel_df[sel_df["dossier_id"]   == dossier_id] if not sel_df.empty  else pd.DataFrame()
        chapters  = []
        for chap_key, chap_rows in doss_rows.groupby("chapter_key", sort=False):
            chap_filt = doss_filt[doss_filt["chapter_key"] == chap_key] if not doss_filt.empty else pd.DataFrame()
            chap_sel  = doss_sel[doss_sel["chapter_key"]   == chap_key] if not doss_sel.empty  else pd.DataFrame()
            chapters.append(_build_chapter(chap_rows.reset_index(drop=True), chap_filt, chap_sel, key_to_name))
        dashboards.append({
            "name":              first["dossier_name"],
            "semantic_model":    semantic_model_name,
            "chapters":          chapters,
            "custom_extensions": [_mstr_ext({"dossier_id": dossier_id})],
        })
    return dashboards




MSTR_VIZ_TYPE_MAP = {
    "Grid":           "table",
    "BarChart":       "bar_chart",
    "LineChart":      "line_chart",
    "PieChart":       "pie_chart",
    "BubbleChart":    "scatter_plot",
    "ComboChart":     "combo_chart",
    "HeatMap":        "heatmap",
    "TreeMap":        "treemap",
    "Selector":       "filter_control",
    "GaugeChart":     "gauge",
    "KpiWidget":      "kpi_card",
    "GeoChart":       "geo_map",
    "WaterfallChart": "waterfall",
    "FunnelChart":    "funnel",
    "PanelStack":     "container",
    "TextBox":        "text_box",
    "Button":         "button",
    "Image":          "image",
}

MSTR_SEL_TYPE_MAP = {
    "attribute_element_list":  "element_selection",
    "metric_qualification":    "expression",
    "object_replacement":      "object_selection",
    "visualization_as_filter": "element_selection",
}



Connection to Strategy One Intelligence Server has been established.
No project selected.


In [ ]:
conn_params =  user_d["conn_params"]
conn = Connection(**conn_params)
conn.headers['Content-type'] = "application/json"

i_doss_read_out     = doss_read_out()
i_doss_read_out_det = doss_read_out_det()

doss_id      = "8D47320148C7E2850BFD2B9605D73297"  # ACBADD1A44EC7158E8D13D930F6DEB46
dossier_id_l = [doss_id]
obj={}
obj["id"]="8D47320148C7E2850BFD2B9605D73297"
obj["type"]="55"
obj["subtype"]="14081"
obj_guid=f"{obj['id']};{obj['type']}"
conn.select_project("B7CA92F04B9FAE8D941C3E9B7E0CD754")

obj_def_d=i_read_gen.get_obj_def(conn=conn,
                                object_id=obj["id"],
                                obj_type=obj["type"],
                                obj_sub_type=obj["subtype"])


doss_hier_l = i_doss_read_out_det.run_read_out_doss_hier_det(conn, dossier_id_l)
filt_sel_d  = i_doss_read_out.run_read_out_doss_filt_sel(conn, dossier_id_l)

# build semantic model + dashboard and merge into one OSI document
osi_d = build_osi_semantic_model(obj_def_d)
osi_d["dashboards"] = build_osi_dashboard(
    doss_hier_l=doss_hier_l,
    filt_sel_d=filt_sel_d,
    semantic_model_name=obj_def_d.get("name", "unknown"),
)

out_path = r"C:\coding\Python_environments\OSI_Files\osi_simple_osi_i.yml"
with open(out_path, "w", encoding="utf-8") as f:
    yaml.dump(osi_d, f, allow_unicode=True, default_flow_style=False, sort_keys=False)
print(f"Written → {out_path}")
osi_d

Connection to Strategy One Intelligence Server has been established.
No project selected.
Written → C:\coding\Python_environments\OSI_Files\osi_simple_osi_i.yml


{'version': '0.2.0',
 'semantic_model': [{'name': 'Regional Marketing Ofensive 2024',
   'datasets': [{'name': 'Onto report',
     'source': '742F452F43CE8C06BDFA34A265AB2E09',
     'custom_extensions': [{'vendor_name': 'MSTR_ROBOTICS',
       'data': '{"dataset_id": "742F452F43CE8C06BDFA34A265AB2E09", "dataset_name": "Onto report"}'}],
     'fields': [{'name': 'Customer Region',
       'expression': {'dialects': [{'dialect': 'MSTR_ROBOTICS',
          'expression': '8D679D3B11D3E4981000E787EC6DE8A4'}]},
       'custom_extensions': [{'vendor_name': 'MSTR_ROBOTICS',
         'data': '{"obj_id": "8D679D3B11D3E4981000E787EC6DE8A4", "obj_type": "attribute", "forms": [{"form_id": "A73FB8C84685F946990E0ABB4CD34123", "form_name": "Cust Region URI", "dataType": "varChar", "baseFormCategory": "Customer Region None (1)", "baseFormType": "url"}, {"form_id": "FE96F95744D713DEA402A69576E7C1DA", "form_name": "Cust Region_onto_id", "dataType": "varChar", "baseFormCategory": "Customer Region None", "b

## Notebook code

## Ontology Mapper

In [ ]:
import pandas as pd
from pathlib import Path


def add_ontology_col(table, primary_key, excel_file, output_path):
    df    = pd.read_excel(excel_file)
    lines = []

    # ── 1. ALTER TABLE ────────────────────────────────────────────────────────
    lines.append(f"-- Add ontology columns to {table}")
    lines.append(f"ALTER TABLE {table} ADD COLUMN ontology_id VARCHAR(500);\n")
    lines.append(f"ALTER TABLE {table} ADD COLUMN ontology_uri VARCHAR(500);\n")

    # ── 2. UPDATE statements ──────────────────────────────────────────────────
    lines.append(f"-- Populate ontology from Wikidata columns")
    for _, row in df.iterrows():
        if pd.isna(row[primary_key]):
            continue
        row_id = int(row[primary_key])

        wiki_id  = str(row["Wikidata_ID"]).strip()
        wiki_uri = str(row["Wikidata_URI"]).strip()
        if wiki_id in ("no direct Wikidata match", "--", "", "nan"):
            wiki_id  = "No Ontology available"
            wiki_uri = "No_ontology"

        ontology_id_escaped = wiki_id.replace("'", "''")
        ontology_uri_escaped = wiki_uri.replace("'", "''")

        lines.append(
            f"UPDATE {table} SET "
            f"ontology_id = '{ontology_id_escaped}' "
            f", ontology_uri = '{ontology_uri_escaped}' "
            f"WHERE {primary_key} = {row_id};"
        )

    sql = "\n".join(lines)
    Path(output_path).write_text(sql, encoding="utf-8")
    print(f"Written {len(df)} UPDATE statements → {output_path}")
    print("\nPreview (first 7 lines):")
    for line in sql.splitlines()[:7]:
        print(" ", line)


# ── run for brands ────────────────────────────────────────────────────────────
add_ontology_col(
    table       = "LU_CUSTOMER",
    primary_key = "zipcode",
    excel_file  = r"C:\Users\danie\Downloads\URI_Zip_Code.xlsx",
    output_path = r"C:\coding\Python_environments\mstr_robotics\import_files\update_zipcode_ontology.sql",
)
